> **Notebook version v1.10.0** · built 2026-09-15 13:43 UTC · git `77cd99b`


# Fire model — scratch campaign (COCO backbone, honest validation)

Trains a fire detector **from a COCO-pretrained backbone** (`yolo11s.pt`) — **not** from the
production fire checkpoint (`models/fire/best.pt`, i.e. v4/v5/v6). Later campaign runs can
continue from the previous run's winner (e.g. `v2` continues `v1/scratch-v1.pt`). Datasets are
downloaded by the notebook itself, deduplicated honestly, and artifacts live on Drive.

- **Dataset build:** download → merge → dedup → clean `train/val/test`. Dedup runs each set
  against ITSELF first, then cross-set (see the honest-validation contract below).
- **Every run dedups against the PREVIOUS runs** via a lightweight fingerprint index on
  Drive (`md5` + 64-bit dHash, ~100 bytes/image). Old datasets are never re-downloaded.
- **CCTV images are test-only** (role=test) and never enter training.
- **Training is detached** (`subprocess.Popen(start_new_session=True)`): closing VS Code /
  the tab cannot stop it, and the per-epoch checkpoint is mirrored to Drive.
- **Incremental improvement:** the COLLECT cell scores the run on the held-out test split,
  compares it to the previous best in `<DRIVE>/registry.json`, and promotes or rejects.

Plan: `plans/fire-model-scratch-colab.md`.


In [ ]:
# Cell 1 - deps + GPU
!nvidia-smi
!pip -q install --upgrade ultralytics pyarrow huggingface_hub gdown
import torch, ultralytics, pyarrow, huggingface_hub
print('ultralytics', ultralytics.__version__, '| pyarrow', pyarrow.__version__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'pick a GPU runtime (Runtime -> Change runtime type)'


In [ ]:
# Cell 2 - EDIT THIS (paths, classes, sources, hyperparameters)

DRIVE_DIR  = '/content/drive/MyDrive/mazr3a-fire-scratch'   # <-- your Drive folder
VERSION    = 'v3'                     # THIS run's campaign folder (v1 = scratch-v1, v2 = dfire, v3 = smoke)
PREV_VERS  = ['v1', 'v2']             # EVERY previous run (base model + merged fingerprints)
UPLOADS     = DRIVE_DIR + '/uploads'                        # shared DATA zips (re-uploaded only when changed)
SCRIPTS_ZIP = DRIVE_DIR + '/' + VERSION + '/scripts.zip'    # per-version scripts (changes every run)
FOLDER_ZIPS = {'negatives': 'negatives.zip', 'domain_test': 'domain_test.zip',
               'cctv_emergency': 'cctv_emergency.zip',
               'salah_haismawi': 'salah_haismawi.zip'}     # local folder -> zip name under UPLOADS/
RUNS       = DRIVE_DIR + '/' + VERSION + '/runs'           # THIS run's artifacts (mirrored every epoch)
FPS_TRAIN  = DRIVE_DIR + '/' + VERSION + '/fingerprints/train'  # THIS run's train index (written by COLLECT)
FPS_TEST   = DRIVE_DIR + '/' + VERSION + '/fingerprints/test'   # THIS run's test/val index (written by COLLECT)
REGISTRY   = DRIVE_DIR + '/' + VERSION + '/registry.json'  # run history + promote/regress decisions
PREV_REG   = DRIVE_DIR + '/' + PREV_VERS[-1] + '/registry.json'  # newest previous run's history
LOGS       = DRIVE_DIR + '/' + VERSION + '/logs'

LOCAL_RUNS = '/content/runs'        # fast local project dir (Drive FUSE is slow)
RUNS_MODE  = 'local_mirror'         # 'local_mirror' (default) | 'drive'
LOG_LOCAL  = '/content/train.log'   # detached trainer stdout (mirrored to Drive)
NAME       = 'scratch-v3'           # run name -> RUNS/<name>/weights/... -> <VERSION>/scratch-v3.pt
PIDFILE    = '/content/train.pid'

# --- VM paths (idempotent) ---
UPLOAD = '/content/upload'          # unpacked scripts + data folders (Cell 4)
CACHE  = '/content/src_cache'       # downloaded sources (temporary)
RAW    = '/content/raw_yolo'        # merged pool (before dedup)
CLEAN  = '/content/clean_yolo'      # clean pool (after dedup) - what we train on

# --- class contract (production order: fire/other/smoke, nc=3) ---
CLASSES   = ['fire', 'other', 'smoke']   # nc=3 - drop-in with models/fire/labelmap.txt
CLASS_MAP = {                           # source class name -> target class (None drops the box)
    'fire': 'fire', 'Fire': 'fire',
    'flame_visible': 'fire',
    'smoke': 'smoke', 'smoke_visible': 'smoke',
    'other': 'other', 'default': 'other',
}

# --- base model (continued FINE-TUNE of run 2's fire-only winner -> head expands nc 1 -> 3) ---
BASE_MODEL = DRIVE_DIR + '/v2/scratch-v1-dfire.pt'   # run 2's fire-only winner (nc=1)
EXPECTED_BASE_MD5 = '3262b25be13b0888d21e7e96d16af20d'  # md5 of the LOCAL v2 copy - launch aborts on mismatch

# --- smaller run: cap the HF FireViewer download (train split only, whole clips) ---
FV_LIMIT = 20000   # FireViewer TRAIN images to keep (group-sampled); ~half of v3 -> ~13 min/epoch

# --- training sources: HF FireViewer + Abonia + SalahALHaismawi + negatives; CCTV is TEST-ONLY ---
SOURCES = [
    # 31 GB HF FireViewer corpus (re-downloaded by Colab; parquet cache freed after conversion).
    # alarmod is GPL-3.0 -> excluded. SMALL RUN: train split only (val/test are already covered
    # by v1/v2's fingerprint index) and capped at FV_LIMIT whole clips; the dedup bg-cap keeps
    # the 60/40 positive/background balance after dedup.
    {'id': 'fireviewer', 'type': 'huggingface_fireviewer',
     'repo': 'fireviewer/fire-smoke-detection-corpus-v1',
     'splits': 'train', 'exclude_sources': ['alarmod'],
     'limit': FV_LIMIT, 'sample_mode': 'group'},
    # Abonia fire-8 (CC BY 4.0), fetched from GitHub via a sparse clone.
    {'id': 'abonia', 'type': 'github_repo',
     'repo': 'Abonia1/YOLOv8-Fire-and-Smoke-Detection',
     'subpath': 'datasets/fire-8', 'role': 'train'},
    # SalahALHaismawi = Roboflow 'Fire Detection.v1i.yolov8'. BUNDLED (the Roboflow URL is
    # Cloudflare-403 for anonymous downloads, so pack_fire_scratch_colab.sh extracts it here).
    {'id': 'salah_haismawi', 'type': 'yolo_dir', 'path': UPLOAD + '/salah_haismawi', 'role': 'train'},
    # domain negatives -> train as background
    {'id': 'negatives', 'type': 'yolo_dir', 'path': UPLOAD + '/negatives', 'role': 'negatives'},
    # our-domain + Simuletic CCTV are held out (never trained, never augmented)
    {'id': 'cctv_test', 'type': 'yolo_dir', 'path': UPLOAD + '/domain_test', 'role': 'test'},
    {'id': 'cctv_emergency', 'type': 'yolo_dir', 'path': UPLOAD + '/cctv_emergency', 'role': 'test'},
]

# --- dedup (strict isolation stays; train duplicates get re-rendered by the augment cell) ---
DEDUP_HAMMING     = 8      # dHash Hamming distance; 6 conservative, 10 aggressive
DEDUP_TRAIN_SCOPE = 'group'  # 'group' = within a clip (video frames); 'global' = whole train set

# --- positive/background balance of TRAIN (empty-label fraction cap) ---
MAX_BG_SHARE = 0.60   # cap the background (empty-label) fraction of TRAIN at 60%
                      #   raise to 0.70-0.80 if false positives dominate in the pilot;
                      #   lower to 0.50 if recall lags. Excess background is moved to
                      #   <clean>_bg_extra (reversible) - never deleted.

# --- augmentation: re-render train duplicates (static, in the augment cell) + online aug ---
AUG_SEED      = 0      # base seed (re-render seeds derive from each stem, so it is stable)
AUG_DEGREES   = 15     # online rotation +-15 deg
AUG_FLIPLR    = 0.5    # online horizontal flip (left<->right)
AUG_FLIPUD    = 0.0    # NO upside-down flip
AUG_HSV_H     = 0.015  # online hue shift
AUG_HSV_S     = 0.5    # online saturation shift
AUG_HSV_V     = 0.1    # online brightness shift +-10%

# --- training (continued low-LR FINE-TUNE of run 2's fire-only winner) ---
EPOCHS      = 40    # smaller pool (~30k) converges in fewer passes; patience 15 early-stops
BATCH       = 16    # 32 on an L4, 64+ on an A100; 8 if a T4 OOMs
IMGSZ       = 640   # matches production input
FREEZE      = 10    # freeze the backbone (fine-tune convention; scratch runs use 0)
LR0         = 0.001 # LOW LR so the model does not forget fire
PATIENCE    = 15    # early stop on the held-out val split
SAVE_PERIOD = 1     # checkpoint every epoch -> survives a recycle

# --- per-cell prerequisite guard (every cell verifies the cells it depends on ran) ---
def need(ok, what):
    if not ok:
        raise SystemExit('PREREQUISITE MISSING - ' + what)

# --- training status (success/failure, not just 'finished?') ---
def training_status():
    """Return (state, done_epochs, log_tail) for the DETACHED trainer.
    state in: none | running | completed | early-stopped | failed | unknown"""
    import os
    rows = []
    for p in (os.path.join(LOCAL_RUNS, NAME, 'results.csv'),
              os.path.join(RUNS, NAME, 'results.csv')):
        if os.path.exists(p):
            rows = [r for r in open(p, errors='replace').read().splitlines() if r.strip()]
            break
    n = max(0, len(rows) - 1)
    pid = int(open(PIDFILE).read().strip() or 0) if os.path.exists(PIDFILE) else 0
    running = False
    if pid and os.path.exists('/proc/%d' % pid):
        try:
            running = open('/proc/%d/stat' % pid).read().split()[2] != 'Z'
        except (FileNotFoundError, IndexError):
            running = False
    log = LOG_LOCAL
    if not os.path.exists(log):
        log = os.path.join(RUNS, NAME, 'train.log')
    tail = ''
    if os.path.exists(log):
        tail = '\n'.join(open(log, errors='replace').read().splitlines()[-25:])
    if running:
        return 'running', n, tail
    if n >= EPOCHS:
        return 'completed', n, tail
    low = tail.lower()
    if ('traceback' in low or 'cuda out of memory' in low
            or 'killed' in low or 'error:' in low):
        return 'failed', n, tail
    if ('earlystop' in low or 'no improvement observed' in low
            or 'stopping training' in low):
        return 'early-stopped', n, tail
    if not rows and pid == 0:
        return 'none', n, tail
    return 'unknown', n, tail


In [ ]:
# Cell 3 - mount Drive + debug logging (every stage's stdout also lands in <DRIVE>/logs/)
import contextlib, os, sys, time
from google.colab import drive

drive.mount('/content/drive')
for d in (DRIVE_DIR, RUNS, FPS_TRAIN, FPS_TEST, LOGS):
    os.makedirs(d, exist_ok=True)

class _Tee:
    """Duplicate everything written to stdout into a file (line-buffered) on Drive."""
    def __init__(self, path):
        self.path = path
        self.f = open(path, 'a', buffering=1)
        self.out = getattr(sys.stdout, 'out', sys.stdout)
    def write(self, s):
        self.f.write(s)
        try:
            self.out.write(s)
        except Exception:
            pass
    def flush(self):
        try:
            self.f.flush()
            self.out.flush()
        except Exception:
            pass
    def close(self):
        try:
            self.f.close()
        except Exception:
            pass

@contextlib.contextmanager
def drive_log(stage):
    """with drive_log('07_dedup'): <work>  -> full stdout also lands in <DRIVE>/logs/."""
    path = os.path.join(LOGS, stage + '.log')
    tee = _Tee(path)
    tee.write('\n' + '=' * 72 + '\n[%s] started %s\n' % (stage, time.strftime('%Y-%m-%d %H:%M:%S'))
              + '=' * 72 + '\n')
    try:
        with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
            yield path
    finally:
        tee.write('[%s] finished %s\n' % (stage, time.strftime('%H:%M:%S')))
        tee.close()
    print('log ->', path)

print('logs ->', LOGS)


In [ ]:
# Cell 4 - unpack the upload bundles from Drive (self-healing + reuses unchanged folders)
import glob, os, zipfile
need(os.path.exists('/content/drive/MyDrive'), 'run Cell 3 (mount Drive) first')

REQUIRED = ['scripts/prep_fire_scratch_dataset.py', 'scripts/prep_fireviewer_dataset.py',
            'scripts/dedup_fire_scratch.py', 'scripts/augment_fire_train.py',
            'scripts/colab_train_scratch.py']
os.makedirs(os.path.join(RUNS, NAME), exist_ok=True)   # the log needs a home on Drive
os.makedirs(UPLOAD, exist_ok=True)

def _unzip(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)

# 1) scripts (per-version: <VERSION>/scripts.zip) - extracted when a required script is missing
if any(not os.path.exists(os.path.join(UPLOAD, r)) for r in REQUIRED):
    assert os.path.exists(SCRIPTS_ZIP), ('missing ' + SCRIPTS_ZIP +
        ' - build with dev_scripts/colab/pack_fire_scratch_colab.sh and upload it to ' + DRIVE_DIR + '/' + VERSION)
    print('extracting scripts ->', UPLOAD)
    _unzip(SCRIPTS_ZIP, UPLOAD)

# 2) shared data folders (<DRIVE>/uploads/<folder>.zip) - reuse an already-present folder
for folder, zip_name in FOLDER_ZIPS.items():
    dst = os.path.join(UPLOAD, folder)
    if os.path.isdir(dst) and any(os.scandir(dst)):
        print('(reuse already-uploaded) %s' % folder)
        continue
    zip_path = os.path.join(UPLOADS, zip_name)
    assert os.path.exists(zip_path), ('missing ' + zip_path + ' - upload it to ' + UPLOADS)
    print('extracting %s -> %s' % (zip_name, dst))
    _unzip(zip_path, UPLOAD)

missing = [r for r in REQUIRED if not os.path.exists(os.path.join(UPLOAD, r))]
assert not missing, ('scripts bundle is missing ' + ', '.join(missing) +
                     ' - rebuild + re-upload scripts.zip to ' + DRIVE_DIR + '/' + VERSION)
print('bundle:', sorted(os.listdir(UPLOAD)))
print('domain_test:', len(glob.glob(UPLOAD + '/domain_test/*')),
      '| negatives:', len(glob.glob(UPLOAD + '/negatives/*')),
      '| salah_haismawi:', len(glob.glob(UPLOAD + '/salah_haismawi/**/*', recursive=True)))


In [ ]:
# Cell 5 - build the RAW merged pool: download external sources -> remap classes -> merge
# IDEMPOTENT: re-running after a reset is a no-op if RAW/data.yaml already exists.
import json, os, subprocess, sys
need(os.path.exists(UPLOAD + '/scripts/prep_fire_scratch_dataset.py'), 'run Cell 4 (unpack bundle) first')
os.environ.setdefault('PYTHONUNBUFFERED', '1')   # stream child stdout line-by-line
def _stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode

with drive_log('05_sources'):
    cfg = {'classes': CLASSES, 'class_map': CLASS_MAP, 'sources': SOURCES}
    cfg_path = '/content/sources.json'
    json.dump(cfg, open(cfg_path, 'w', encoding='utf-8'), indent=2)
    prep = UPLOAD + '/scripts/prep_fire_scratch_dataset.py'
    assert os.path.exists(prep), 'bundle missing prep script - see Cell 4'
    rc = _stream([sys.executable, prep, '--config', cfg_path, '--out', RAW,
                  '--cache', CACHE, '--scripts', UPLOAD + '/scripts'])
    if rc != 0:
        raise SystemExit('prep_fire_scratch_dataset.py exited ' + str(rc))
    print(open(RAW + '/prep_report.txt', encoding='utf-8').read())
    import shutil as _sh
    _sh.rmtree(CACHE, ignore_errors=True)   # drop the downloaded source (saves ~25-31 GB)
    print('freed source cache ->', CACHE)


In [ ]:
# Cell 6 - dedup (isolation only) + augment ALL train images
# dedup now SKIPS the train-side near-dup scan (--no-train-self, no --prev-train-index) so no
# positive train signal is thrown away; it only enforces test/val isolation. Then
# augment_fire_train.py re-renders EVERY train image. Test/val stay byte-original.
# Resumable/reusable: dedup + augment checkpoints are mirrored to <DRIVE>/<VERSION>/state/ and
# restored at the start (gated by the sources config md5) so a recycle resumes, not restarts.
import os, subprocess, sys, shutil as _sh
# concurrent workers for the dedup + augment steps (I/O + C-extension bound; threads overlap)
WORKERS = min(16, max(2, (os.cpu_count() or 2) * 2))
need(os.path.exists(RAW + '/data.yaml'), 'run Cell 5 (build raw pool) first')
need(os.path.exists(UPLOAD + '/scripts/dedup_fire_scratch.py'), 'run Cell 4 (unpack bundle) first')
need(os.path.exists(UPLOAD + '/scripts/augment_fire_train.py'), 'run Cell 4 (unpack bundle) first')
os.environ.setdefault('PYTHONUNBUFFERED', '1')   # stream child stdout line-by-line
def _stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode

with drive_log('06_dedup'):
    os.makedirs(FPS_TRAIN, exist_ok=True)
    os.makedirs(FPS_TEST, exist_ok=True)

    # --- restore last run's reusable state from Drive so a recycle resumes, not restarts ---
    STATE_DIR = DRIVE_DIR + '/' + VERSION + '/state'
    import hashlib as _hl
    _cfg_hash = (_hl.md5(open('/content/sources.json', 'rb').read()).hexdigest()
                 if os.path.exists('/content/sources.json') else '')
    _gate = os.path.join(STATE_DIR, 'sources.md5')
    if _cfg_hash and os.path.isfile(_gate) and open(_gate).read().strip() == _cfg_hash:
        os.makedirs(CLEAN + '_report', exist_ok=True)
        for _f in ('fingerprints.jsonl', 'augmentation_state.jsonl'):
            _s = os.path.join(STATE_DIR, _f)
            if os.path.exists(_s):
                _sh.copy2(_s, os.path.join(CLEAN + '_report', _f))
        print('restored dedup/augment state from Drive ->', CLEAN + '_report')


    # merge EVERY previous run's held-out TEST/val fingerprints (isolation) into one local dir
    prev_test = '/content/prev_test_fp'
    for v in PREV_VERS:
        src = DRIVE_DIR + '/' + v + '/fingerprints/test'
        if os.path.isdir(src):
            os.makedirs(prev_test, exist_ok=True)
            for f in os.listdir(src):
                if f.endswith('.jsonl'):
                    _sh.copy2(os.path.join(src, f), os.path.join(prev_test, v + '__' + f))

    d = UPLOAD + '/scripts/dedup_fire_scratch.py'
    rc = _stream([sys.executable, d,
                  '--pool', RAW, '--out', CLEAN,
                  '--hamming', str(DEDUP_HAMMING),
                  '--train-scope', DEDUP_TRAIN_SCOPE,
                  '--max-bg-share', str(MAX_BG_SHARE),
                  '--prev-test-index', prev_test,
                  '--no-train-self',
                  '--run-name', NAME,
                  '--report', CLEAN + '_report',
                  '--workers', str(WORKERS),
                  '--skip-broken', '--broken-out', '/content/broken'])
    if rc != 0:
        raise SystemExit('dedup_fire_scratch.py exited ' + str(rc))
    print(open(CLEAN + '_report/summary.txt', encoding='utf-8').read())
    _sh.rmtree(RAW, ignore_errors=True)   # dedup done; the raw pool is no longer needed
    print('freed raw pool ->', RAW)

    # augment EVERY train image (rotation/brightness/contrast/noise/hue/flip); test/val untouched
    a = UPLOAD + '/scripts/augment_fire_train.py'
    rc = _stream([sys.executable, a,
                  '--clean', CLEAN, '--report', CLEAN + '_report',
                  '--workers', str(WORKERS)])
    if rc != 0:
        raise SystemExit('augment_fire_train.py exited ' + str(rc))
    print(open(CLEAN + '_report/augmentation_summary.txt', encoding='utf-8').read())

    # --- persist the reusable state to Drive: keep/discard decisions + per-image aug ops ---
    os.makedirs(STATE_DIR, exist_ok=True)
    for _f in ('fingerprints.jsonl', 'decisions.jsonl', 'per_image.csv',
               'augmentation_state.jsonl', 'augmentation_report.csv'):
        _s = os.path.join(CLEAN + '_report', _f)
        if os.path.exists(_s):
            _sh.copy2(_s, os.path.join(STATE_DIR, _f))
    if _cfg_hash:
        open(os.path.join(STATE_DIR, 'sources.md5'), 'w').write(_cfg_hash)
    print('dedup/augment state persisted ->', STATE_DIR)


In [ ]:
# Cell 7 - verify the CLEAN pool (read-only): splits, class coverage, integrity
import collections, os
need(os.path.exists(CLEAN + '/data.yaml'), 'run Cell 6 (dedup) first')

EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
with drive_log('07_verify'):
    for split in ('train', 'val', 'test'):
        # train was augmented into the train_aug split (literal images/labels names)
        if split == 'train':
            idir = os.path.join(CLEAN, 'train_aug', 'images')
            ldir = os.path.join(CLEAN, 'train_aug', 'labels')
        else:
            idir = os.path.join(CLEAN, split, 'images')
            ldir = os.path.join(CLEAN, split, 'labels')
        if not os.path.isdir(idir):
            print('[%s] (absent)' % split)
            continue
        imgs = [f for f in os.listdir(idir) if f.lower().endswith(EXT)]
        cls = collections.Counter()
        missing = empty = 0
        for f in imgs:
            lb = os.path.join(ldir, os.path.splitext(f)[0] + '.txt')
            if not os.path.isfile(lb):
                missing += 1
                continue
            if os.path.getsize(lb) == 0:
                empty += 1
            for row in open(lb, encoding='utf-8'):
                parts = row.split()
                if parts:
                    cls[int(float(parts[0]))] += 1
        names = {i: n for i, n in enumerate(CLASSES)}
        print('[%s] images=%d missing_label=%d background=%d boxes=%s' % (split, len(imgs), missing, empty,
                 ', '.join('%s=%d' % (names.get(k, k), v) for k, v in sorted(cls.items())) or '-'))
    # positive (any class) vs background balance of the CLEAN train split
    _ti, _tl = os.path.join(CLEAN, 'train_aug', 'images'), os.path.join(CLEAN, 'train_aug', 'labels')
    _npos = _nbg = 0
    for _f in os.listdir(_ti):
        _lb = os.path.join(_tl, os.path.splitext(_f)[0] + '.txt')
        if os.path.isfile(_lb) and os.path.getsize(_lb) > 0:
            _npos += 1
        else:
            _nbg += 1
    print('\ntrain balance: positive(any class)=%d (%.0f%%)  background=%d (%.0f%%)  cap=%.0f%%' % (_npos, 100 * _npos / max(1, _npos + _nbg),
             _nbg, 100 * _nbg / max(1, _npos + _nbg), 100 * MAX_BG_SHARE))
    print('\ndata.yaml:')
    print(open(CLEAN + '/data.yaml', encoding='utf-8').read().strip())
    import shutil as _sh
    print('\n/content free: %.1f GB' % (_sh.disk_usage('/content').free / 1e9))


In [ ]:
# Cell 8 - sanity-check the detached trainer that ships in the bundle
import os, subprocess, sys
need(os.path.exists(UPLOAD + '/scripts/colab_train_scratch.py'), 'run Cell 4 (unpack bundle) first')
trainer = UPLOAD + '/scripts/colab_train_scratch.py'
print(subprocess.run([sys.executable, trainer, '--help'],
                     capture_output=True, text=True).stdout[:900])


In [ ]:
# Cell 9 - LAUNCH training DETACHED (closing VS Code will NOT stop it)
import hashlib, os, subprocess, sys
need(os.path.exists(CLEAN + '/data.yaml'), 'run Cell 6 (dedup) first')
need(os.path.exists(UPLOAD + '/scripts/colab_train_scratch.py'), 'run Cell 4 (unpack bundle) first')

trainer = UPLOAD + '/scripts/colab_train_scratch.py'   # defined here too: Cell 8 is optional
assert os.path.exists(trainer), 'trainer missing from the bundle - see Cell 4'

# pre-flight: verify the Drive base model is the EXACT run-1 winner (avoid fine-tuning the wrong model)
_base_md5 = hashlib.md5(open(BASE_MODEL, 'rb').read()).hexdigest()
assert _base_md5 == EXPECTED_BASE_MD5, ('BASE_MODEL md5 MISMATCH: expected %s, got %s '
                                        '- do NOT train on the wrong model' % (EXPECTED_BASE_MD5, _base_md5))
print('base model md5 OK:', _base_md5)

if os.path.exists(PIDFILE):
    _old = int(open(PIDFILE).read().strip() or 0)
    try:
        os.kill(_old, 0)
        raise SystemExit('a trainer is already running (pid %d). Watch it with Cell 10, '
                         'or kill it first: !kill -9 %d' % (_old, _old))
    except OSError:
        pass

os.makedirs(os.path.join(RUNS, NAME), exist_ok=True)
args = [sys.executable, '-u', trainer,
        '--data', CLEAN + '/data.yaml', '--model', BASE_MODEL,
        '--runs', RUNS, '--name', NAME, '--mode', RUNS_MODE, '--local', LOCAL_RUNS,
        '--log', LOG_LOCAL,
        '--epochs', str(EPOCHS), '--batch', str(BATCH), '--imgsz', str(IMGSZ),
        '--freeze', str(FREEZE), '--lr0', str(LR0), '--patience', str(PATIENCE),
        '--degrees', str(AUG_DEGREES), '--fliplr', str(AUG_FLIPLR), '--flipud', str(AUG_FLIPUD),
        '--hsv-h', str(AUG_HSV_H), '--hsv-s', str(AUG_HSV_S), '--hsv-v', str(AUG_HSV_V),
        '--save-period', str(SAVE_PERIOD)]
with open(LOG_LOCAL, 'ab') as lf:
    proc = subprocess.Popen(args, cwd='/content', stdout=lf,
                            stderr=subprocess.STDOUT, start_new_session=True)
open(PIDFILE, 'w').write(str(proc.pid))
print('launched pid %d (own session -> survives losing the editor/kernel)' % proc.pid)
print('local log :', LOG_LOCAL)
print('Drive run :', os.path.join(RUNS, NAME), '(mirrored every epoch)')
print('next      : run Cell 10 to watch progress')


In [ ]:
# Cell 10 - STATUS: re-attach to the detached run (no stdout stream needed)
import os, subprocess, time

with drive_log('10_status'):
    need(os.path.exists(PIDFILE) or
         os.path.exists(os.path.join(LOCAL_RUNS, NAME, 'results.csv')) or
         os.path.exists(os.path.join(RUNS, NAME, 'results.csv')),
         'no training launched - run Cell 9 (LAUNCH) first')
    st, done, tail = training_status()
    print('training status: %s (%d/%d epochs)' % (st, done, EPOCHS))
    if st == 'failed':
        print('-- log tail (failure) --')
        print(tail)

    pid = int(open(PIDFILE).read().strip() or 0) if os.path.exists(PIDFILE) else None

    def proc_state(p):
        if not p:
            return 'no pid file'
        try:
            with open('/proc/%d/stat' % p) as fh:
                fields = fh.read().split()
            state = fields[2]
        except (FileNotFoundError, IndexError):
            return 'not running (no such process)'
        if state == 'Z':
            return 'ZOMBIE (exited - read the log tail below)'
        return {'R': 'RUNNING', 'S': 'RUNNING (sleeping)', 'D': 'RUNNING (disk-sleep)',
                'T': 'stopped'}.get(state, state)

    print('trainer pid %s -> %s' % (pid, proc_state(pid)))
    log_age = '%.0fs' % (time.time() - os.path.getmtime(LOG_LOCAL)) if os.path.exists(LOG_LOCAL) else 'n/a'
    print('local log : %s (%d B, written %s ago)' % (LOG_LOCAL,
          os.path.getsize(LOG_LOCAL) if os.path.exists(LOG_LOCAL) else 0, log_age))
    run_local = os.path.join(LOCAL_RUNS, NAME)
    run_drive = os.path.join(RUNS, NAME)

    for label, d in (('local', run_local), ('Drive', run_drive)):
        print('\n[%s] %s' % (label, d))
        if not os.path.isdir(d):
            print('   (missing)')
            continue
        csvp = os.path.join(d, 'results.csv')
        if os.path.exists(csvp):
            rows = [r for r in open(csvp).read().splitlines() if r.strip()]
            print('   results.csv  epochs=%d' % max(0, len(rows) - 1))
            if len(rows) > 1:
                print('   last row    :', rows[-1][:140])
        w = os.path.join(d, 'weights')
        if os.path.isdir(w):
            for f in sorted(os.listdir(w)):
                p = os.path.join(w, f)
                print('   %-9s %7.1f MB' % (f, os.path.getsize(p) / 1e6))

    print('\n-- local log tail --')
    if os.path.exists(LOG_LOCAL):
        print('\n'.join(open(LOG_LOCAL, errors='replace').read().splitlines()[-15:]))
    else:
        print('(no log yet)')

    print('\n-- GPU --')
    print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                          '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip() or '(n/a)')


In [ ]:
# Cell 11 - WAIT for the run to finish (OPTIONAL; safe to interrupt)
import os, time

WAIT_POLL_S  = 60      # seconds between polls
WAIT_MAX_MIN = 480     # stop waiting after this long (training keeps running)

run_local = os.path.join(LOCAL_RUNS, NAME)
run_drive = os.path.join(RUNS, NAME)
started = (os.path.exists(PIDFILE) or
           os.path.exists(os.path.join(run_local, 'results.csv')) or
           os.path.exists(os.path.join(run_drive, 'results.csv')))
need(started, 'no training launched - run Cell 9 (LAUNCH) first')

t0 = time.time(); last = -1
while True:
    st, n, tail = training_status()
    if n != last:
        print('[%s] %s %d/%d epochs | waiting %d min' % (time.strftime('%H:%M:%S'), st, n, EPOCHS,
                 int((time.time() - t0) / 60)), flush=True)
        last = n
    if st in ('completed', 'early-stopped'):
        print('-> training finished successfully (%s).' % st)
        break
    if st == 'failed':
        print('-> training FAILED. log tail:')
        print(tail)
        break
    if st in ('none', 'unknown'):
        print('-> no live trainer (%s). Check Cell 10, then resume with Cell 12.' % st)
        break
    if (time.time() - t0) > WAIT_MAX_MIN * 60:
        print('WAIT_MAX_MIN reached - no longer waiting (training continues detached)')
        break
    time.sleep(WAIT_POLL_S)


In [ ]:
# Cell 12 - RESUME an interrupted run (detached too). Refuses if the run FINISHED.
import os, shutil, subprocess, sys, torch
need(os.path.exists(UPLOAD + '/scripts/colab_train_scratch.py'), 'run Cell 4 (unpack bundle) first')

trainer = UPLOAD + '/scripts/colab_train_scratch.py'
run_local, run_drive = os.path.join(LOCAL_RUNS, NAME), os.path.join(RUNS, NAME)
if not os.path.isdir(run_local) and os.path.isdir(run_drive):
    shutil.copytree(run_drive, run_local, dirs_exist_ok=True)
    print('restored run dir from Drive ->', run_local)

ck = os.path.join(run_local, 'weights', 'last.pt')
if not os.path.exists(ck):
    ck = os.path.join(run_drive, 'weights', 'last.pt')
print('checkpoint:', ck)
assert os.path.exists(ck), 'no last.pt - run Cell 9 first'

meta = torch.load(ck, map_location='cpu', weights_only=False)
ta = meta.get('train_args') or {}
done, total = int(meta.get('epoch', -1)) + 1, int(ta.get('epochs', 0) or 0)
resumable = meta.get('optimizer') is not None
print('epoch %d/%d - optimizer %s' % (done, total, 'present' if resumable else 'STRIPPED (finished)'))
if not resumable:
    print('=> nothing to resume: this run is COMPLETE. Use Cell 13 to collect best.pt.')
else:
    with open(LOG_LOCAL, 'ab') as lf:
        proc = subprocess.Popen([sys.executable, '-u', trainer, '--resume', ck],
                                cwd='/content', stdout=lf, stderr=subprocess.STDOUT,
                                start_new_session=True)
    open(PIDFILE, 'w').write(str(proc.pid))
    print('resumed detached, pid', proc.pid, '-> watch with Cell 10')


In [ ]:
# Cell 13 - COLLECT + EVALUATE on the held-out test split + compare vs previous best
# (BLOCKS until the detached trainer launched by Cell 9 has finished)
import hashlib, json, os, shutil, time
need(os.path.exists(CLEAN + '/data.yaml'), 'run Cell 6 (dedup) first')
from ultralytics import YOLO

run_local = os.path.join(LOCAL_RUNS, NAME)
run_drive = os.path.join(RUNS, NAME)
have_ckpt = (os.path.exists(os.path.join(run_local, 'weights', 'best.pt')) or
             os.path.exists(os.path.join(run_drive, 'weights', 'best.pt')) or
             os.path.exists(os.path.join(run_local, 'weights', 'last.pt')) or
             os.path.exists(os.path.join(run_drive, 'weights', 'last.pt')))
need(os.path.exists(PIDFILE) or
     os.path.exists(os.path.join(run_local, 'results.csv')) or
     os.path.exists(os.path.join(run_drive, 'results.csv')) or have_ckpt,
     'run Cell 9 (LAUNCH) first')

# 1) WAIT for the detached trainer to finish (Cell 9 returns immediately)
WAIT_POLL_S = 30
WAIT_MAX_MIN = 480
def _trainer_state():
    pid = int(open(PIDFILE).read().strip() or 0) if os.path.exists(PIDFILE) else 0
    if pid and os.path.exists('/proc/%d' % pid):
        try:
            s = open('/proc/%d/stat' % pid).read().split()[2]
        except (FileNotFoundError, IndexError):
            s = 'Z'
        return 'running' if s != 'Z' else 'zombie'
    return 'gone' if pid else 'none'

t0 = time.time(); n = 0; st = 'none'
while True:
    rows = []
    for p in (os.path.join(run_local, 'results.csv'), os.path.join(run_drive, 'results.csv')):
        if os.path.exists(p):
            rows = [r for r in open(p, errors='replace').read().splitlines() if r.strip()]
            break
    n = max(0, len(rows) - 1)
    st = _trainer_state()
    if st == 'none' or n >= EPOCHS or st in ('gone', 'zombie'):
        break
    if (time.time() - t0) > WAIT_MAX_MIN * 60:
        print('WAIT_MAX_MIN reached - collecting what exists (training may still be running)')
        break
    print('[%s] waiting for training ... %d/%d epochs (trainer %s)' % (time.strftime('%H:%M:%S'), n, EPOCHS, st), flush=True)
    time.sleep(WAIT_POLL_S)
print('training finished: %d epoch row(s), trainer %s' % (n, st))
_st, _n, _tail = training_status()
if _st == 'failed':
    print('\nTRAINING FAILED - log tail:')
    print(_tail)
    raise SystemExit('training failed - fix the issue, then resume (Cell 12) or re-launch (Cell 9)')
if _st in ('completed', 'early-stopped'):
    print('training %s - collecting the best checkpoint.' % _st)

# 2) locate the best checkpoint (local first, then Drive)
src = None
for cand in (os.path.join(LOCAL_RUNS, NAME, 'weights', 'best.pt'),
             os.path.join(RUNS, NAME, 'weights', 'best.pt'),
             os.path.join(LOCAL_RUNS, NAME, 'weights', 'last.pt'),
             os.path.join(RUNS, NAME, 'weights', 'last.pt')):
    if os.path.exists(cand):
        src = cand
        break
assert src, 'no checkpoint found - did Cell 9 run?'
out_pt = os.path.join(DRIVE_DIR, VERSION, NAME + '.pt')
shutil.copy2(src, out_pt)
md5 = hashlib.md5(open(out_pt, 'rb').read()).hexdigest()
print('candidate ->', out_pt)
print('md5: %s | bytes: %d' % (md5, os.path.getsize(out_pt)))

# 3) evaluate on the held-out test split (val points at test/images)
test_yaml = CLEAN + '/data_test.yaml'
lines = [ln for ln in open(CLEAN + '/data.yaml', encoding='utf-8') if not ln.startswith('val:')]
lines.append('val: test/images\n')
open(test_yaml, 'w', encoding='utf-8').writelines(lines)
m = YOLO(out_pt)
res = m.val(data=test_yaml, imgsz=IMGSZ, batch=BATCH, device=0, verbose=False)
metrics = {
    'map50': round(float(res.box.map50), 4),
    'map50_95': round(float(res.box.map), 4),
    'precision': round(float(res.box.mp), 4),
    'recall': round(float(res.box.mr), 4),
    'per_class_map50': [round(float(x), 4) for x in (res.box.maps or [])],
}
print('classes:', list(m.names.values()))
print('test metrics:', metrics)

# 4) registry: append this run, then compare vs the previous best on mAP@50
entry = {
    'name': NAME,
    'date': time.strftime('%Y-%m-%d %H:%M:%S UTC'),
    'classes': CLASSES,
    'base_model': BASE_MODEL,
    'md5': md5,
    'bytes': os.path.getsize(out_pt),
    'metrics': metrics,
    'metric_key': 'map50',
}
registry = []
if os.path.exists(REGISTRY):
    registry = json.load(open(REGISTRY, encoding='utf-8'))
elif os.path.exists(PREV_REG):
    registry = json.load(open(PREV_REG, encoding='utf-8'))   # chain the previous run's history
registry.append(entry)
json.dump(registry, open(REGISTRY, 'w', encoding='utf-8'), indent=2)

prev = registry[:-1]
best_prev = max(prev, key=lambda e: e['metrics']['map50']) if prev else None
cur = metrics['map50']
if best_prev is None:
    decision = 'FIRST RUN - promoted'
    active = entry
elif cur > best_prev['metrics']['map50']:
    decision = 'PROMOTED (map50 %.4f > previous best %.4f)' % (cur, best_prev['metrics']['map50'])
    active = entry
else:
    decision = 'REGRESSED (map50 %.4f <= previous best %.4f) - previous best stays ACTIVE' % (cur, best_prev['metrics']['map50'])
    active = best_prev
json.dump({'active': active['name']}, open(DRIVE_DIR + '/active.json', 'w', encoding='utf-8'), indent=2)
print('\nDECISION:', decision)
print('ACTIVE  :', active['name'])
if best_prev is not None and cur <= best_prev['metrics']['map50']:
    print('\nREGRESSION - decide (human call, nothing is silently overwritten):')
    print('  (a) RETRAIN the previous version - edit NAME + hyperparameters and run Cells 5-9 again, or')
    print('  (b) CONTINUE with this regressed one - if its failure mode (e.g. fewer false positives)')
    print('      is what you need despite the lower mAP, keep ' + NAME + ' as the base for the next run.')

# 5) promote THIS run's lightweight fingerprints to the persistent Drive index
#    (only now - training succeeded - so the index accumulates exactly the trained runs)
for f, dst_dir in (('run_train_index.jsonl', FPS_TRAIN), ('run_test_index.jsonl', FPS_TEST)):
    p = CLEAN + '_report/' + f
    if os.path.exists(p):
        shutil.copy2(p, os.path.join(dst_dir, NAME + '.jsonl'))
        print('fingerprint index updated:', os.path.join(dst_dir, NAME + '.jsonl'))
print('\nregistry ->', REGISTRY)


## Next steps

1. Pull the candidate `DRIVE_DIR/VERSION/<NAME>.pt` from Drive back into the repo and judge it
   **locally** against the full matrix (`dev_scripts/fire/compare_fire_models.py`) and the on-domain
   CCTV audit (`dev_scripts/fire/test_fire_model.py` on the held-out evidence / `domain_test`).
2. If it is a genuine win, convert + archive it per
   [`plans/model-versioning.md`](../plans/model-versioning.md):
   `prep_fire_model.sh` (writes `labelmap.txt` = your `CLASSES`) -> `promote_fire_model.sh` ->
   on-host `firewatch.py --dry-run` pilot (the real acceptance test).
3. For the **next run**, bump `VERSION` (e.g. `v4`) and append to `PREV_VERS` (e.g. `['v1','v2','v3']`);
   the new set is automatically deduped against EVERY previous run via its merged fingerprint dirs.
4. Per-stage logs live on Drive at `DRIVE_DIR/VERSION/logs/`.
